In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to the pre-processed SMARD dataset
file_path = (
    "/content/drive/MyDrive/Colab Notebooks/SMARD_Cleaned_20260806_20260816.csv"
)

# Verify file existence
if os.path.exists(file_path):
  print("Dataset found successfully! Proceeding with data loading...")
else:
  print(
      "File not found! Please verify the folder structure in your Google Drive."
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully! Proceeding with data loading...


In [3]:
# ==========================================
# GERMAN BESS CO-OPTIMIZATION & GRID FEE ENGINE
# ==========================================

import numpy as np
import pandas as pd
import pulp as plp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------
# 1. GENERATE SYNTHETIC GERMAN MARKET DATA
# ------------------------------------------
def generate_german_market_data():
    """
    Generates a 24-hour sample dataset including EPEX Day-Ahead prices,
    German aFRR capacity prices, and industrial load profiles for peak shaving.
    """
    np.random.seed(42)
    hours = pd.date_range(start="2026-06-01", periods=24, freq="h")

    # EPEX Day-Ahead prices (€/MWh) with typical duck-curve / peak behaviors
    da_prices = 80 + 40 * np.sin(np.linspace(0, 2 * np.pi, 24)) + np.random.normal(0, 15, 24)
    da_prices[18:21] = da_prices[18:21] + 120  # Evening price spike

    # aFRR capacity prices (€/MW/h)
    afrr_cap_prices = 25 + 10 * np.abs(np.cos(np.linspace(0, 2 * np.pi, 24))) + np.random.normal(0, 3, 24)

    # Industrial load profile (MW) exhibiting high peaks during midday/evening
    industrial_load = 4.0 + 1.5 * np.sin(np.linspace(0, 2 * np.pi, 24)) + np.random.normal(0, 0.2, 24)
    industrial_load[10:14] = industrial_load[10:14] + 2.5  # Industrial peak hours for grid fees

    df = pd.DataFrame({
        "Timestamp": hours,
        "DA_Price": da_prices,
        "aFRR_Capacity_Price": afrr_cap_prices,
        "Industrial_Load_MW": industrial_load
    })
    df.set_index("Timestamp", inplace=True)
    return df

# Load market dataset
market_df = generate_german_market_data()
print("Market Data Sample:")
display(market_df.head())

# ------------------------------------------
# 2. MILP OPTIMIZATION ENGINE (PuLP & CBC)
# ------------------------------------------
def run_german_bess_optimization(df, capacity_mwh=10.0, max_power_mw=5.0, efficiency=0.9):
    """
    Solves the Mixed-Integer Linear Programming (MILP) co-optimization problem
    for German Day-Ahead arbitrage, aFRR capacity reservation, and Peak Shaving.
    """
    model = plp.LpProblem("German_BESS_CoOptimization", plp.LpMaximize)

    time_steps = range(len(df))

    # Decision Variables
    charge = {t: plp.LpVariable(f"charge_{t}", lowBound=0, upBound=max_power_mw) for t in time_steps}
    discharge = {t: plp.LpVariable(f"discharge_{t}", lowBound=0, upBound=max_power_mw) for t in time_steps}
    reserve_afrr = {t: plp.LpVariable(f"reserve_afrr_{t}", lowBound=0, upBound=max_power_mw/2) for t in time_steps}
    soc = {t: plp.LpVariable(f"soc_{t}", lowBound=0, upBound=capacity_mwh) for t in time_steps}

    # Binary variables to prevent simultaneous high-intensity charging and discharging
    is_charging = {t: plp.LpVariable(f"is_charging_{t}", cat='Binary') for t in time_steps}

    # Objective Function: Maximize total revenue (DA Energy + aFRR Capacity)
    da_prices = df["DA_Price"].values
    afrr_prices = df["aFRR_Capacity_Price"].values

    revenue_expr = plp.lpSum([
        (discharge[t] - charge[t]) * da_prices[t] + reserve_afrr[t] * afrr_prices[t]
        for t in time_steps
    ])
    model += revenue_expr

    # Constraints
    soc_init = capacity_mwh * 0.5  # Initial State of Charge at 50%

    for t in time_steps:
        # Power and reserve boundary condition: Total power allocation cannot exceed inverter limits
        model += charge[t] + reserve_afrr[t] <= max_power_mw
        model += discharge[t] + reserve_afrr[t] <= max_power_mw

        # Mutual exclusivity constraint via binary variables
        model += charge[t] <= max_power_mw * is_charging[t]
        model += discharge[t] <= max_power_mw * (1 - is_charging[t])

        # State of Charge (SoC) dynamics
        if t == 0:
            model += soc[t] == soc_init + (charge[t] * efficiency - discharge[t] / efficiency) * 1.0
        else:
            model += soc[t] == soc[t-1] + (charge[t] * efficiency - discharge[t] / efficiency) * 1.0

    # Final boundary condition for SoC
    model += soc[len(time_steps)-1] >= soc_init

    # Solve the optimization problem using CBC solver
    model.solve(plp.PULP_CBC_CMD(msg=0))

    # Extract results into a structured DataFrame
    results_df = df.copy()
    results_df["Optimized_Charge_MW"] = [plp.value(charge[t]) for t in time_steps]
    results_df["Optimized_Discharge_MW"] = [plp.value(discharge[t]) for t in time_steps]
    results_df["Optimized_Reserve_MW"] = [plp.value(reserve_afrr[t]) for t in time_steps]
    results_df["Optimized_SoC_MWh"] = [plp.value(soc[t]) for t in time_steps]

    return results_df

# Execute optimization
optimized_results = run_german_bess_optimization(market_df)
print("Optimization Status: Successful")

# ------------------------------------------
# 3. FINANCIAL METRICS & VISUALIZATION
# ------------------------------------------
# Calculate financial key performance indicators (KPIs)
total_da_rev = ((optimized_results['Optimized_Discharge_MW'] - optimized_results['Optimized_Charge_MW']) * optimized_results['DA_Price']).sum()
total_afrr_rev = (optimized_results['Optimized_Reserve_MW'] * optimized_results['aFRR_Capacity_Price']).sum()
total_net_revenue = total_da_rev + total_afrr_rev

print(f"--- FINANCIAL SUMMARY ---")
print(f"Day-Ahead Net Revenue: €{total_da_rev:,.2f}")
print(f"aFRR Capacity Revenue: €{total_afrr_rev:,.2f}")
print(f"Total Co-Optimization Revenue: €{total_net_revenue:,.2f}")

# Plotting with Plotly (Dark Mode & High Contrast)
fig = make_subplots(specs=[[{"secondary_y": True}]])
hours = optimized_results.index

fig.add_trace(go.Bar(x=hours, y=optimized_results['Optimized_Charge_MW'], name="Charge (MW)", marker_color='#00CC96'), secondary_y=False)
fig.add_trace(go.Bar(x=hours, y=-optimized_results['Optimized_Discharge_MW'], name="Discharge (MW)", marker_color='#EF553B'), secondary_y=False)
fig.add_trace(go.Bar(x=hours, y=optimized_results['Optimized_Reserve_MW'], name="aFRR Reserve (MW)", marker_color='#AB63FA'), secondary_y=False)
fig.add_trace(go.Scatter(x=hours, y=optimized_results['DA_Price'], name="DA Price (€/MWh)", line=dict(color='#FFA15A', dash='dot')), secondary_y=True)

fig.update_layout(
    title="German BESS Co-Optimization Dispatch Profile (EPEX DA & aFRR)",
    template="plotly_dark",
    barmode='relative',
    hovermode="x unified",
    legend=dict(bgcolor="#581c87", bordercolor="#9333ea", borderwidth=1)
)

fig.show()

Market Data Sample:


,DA_Price,aFRR_Capacity_Price,Industrial_Load_MW
Timestamp,,,
2026-06-01 00:00:00,87.450712,33.366852,4.068724
2026-06-01 01:00:00,88.717906,34.961941,4.052087
2026-06-01 02:00:00,110.498686,30.091213,4.844193
2026-06-01 03:00:00,132.078886,32.952625,5.019237
2026-06-01 04:00:00,112.003108,27.798734,5.196443


Optimization Status: Successful
--- FINANCIAL SUMMARY ---
Day-Ahead Net Revenue: €1,568.72
aFRR Capacity Revenue: €1,750.46
Total Co-Optimization Revenue: €3,319.18


In [4]:
# ==========================================
# PHASE 3: GRID FEE AVOIDANCE & PEAK SHAVING MODULE
# ==========================================

def calculate_peak_shaving_impact(results_df, peak_threshold_mw=5.0):
    """
    Calculates the peak shaving effect of the BESS on an industrial load profile.
    It identifies hours where industrial load exceeds the threshold and
    discharges the battery to cap the grid-visible peak, reducing grid fees.
    """
    df = results_df.copy()

    # Calculate net load seen by the grid after BESS intervention
    # When BESS discharges, it reduces the load drawn from the public grid
    df["Net_Grid_Load_MW"] = df["Industrial_Load_MW"] - df["Optimized_Discharge_MW"] + df["Optimized_Charge_MW"]

    # Identify peak periods exceeding the threshold
    original_peak = df["Industrial_Load_MW"].max()
    optimized_peak = df["Net_Grid_Load_MW"].max()

    peak_reduction = original_peak - optimized_peak

    print(f"--- PEAK SHAVING PERFORMANCE ---")
    print(f"Original Peak Industrial Load: {original_peak:.2f} MW")
    print(f"Optimized Net Peak Load (Grid-Visible): {optimized_peak:.2f} MW")
    print(f"Peak Reduction Achieved: {peak_reduction:.2f} MW")

    return df

# Apply Peak Shaving logic to our optimized results
peak_shaved_results = calculate_peak_shaving_impact(optimized_results)

--- PEAK SHAVING PERFORMANCE ---
Original Peak Industrial Load: 7.16 MW
Optimized Net Peak Load (Grid-Visible): 8.37 MW
Peak Reduction Achieved: -1.20 MW


In [5]:
# ==========================================
# GERMAN BESS TRUE CO-OPTIMIZATION & PEAK SHAVING ENGINE
# ==========================================

import numpy as np
import pandas as pd
import pulp as plp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------
# 1. GENERATE SYNTHETIC GERMAN MARKET & LOAD DATA
# ------------------------------------------
def generate_german_market_data():
    """
    Generates a 24-hour sample dataset including EPEX Day-Ahead prices,
    German aFRR capacity prices, and industrial load profiles for peak shaving.
    """
    np.random.seed(42)
    hours = pd.date_range(start="2026-06-01", periods=24, freq="h")

    # EPEX Day-Ahead prices (€/MWh) with typical duck-curve / peak behaviors
    da_prices = 80 + 40 * np.sin(np.linspace(0, 2 * np.pi, 24)) + np.random.normal(0, 15, 24)
    da_prices[18:21] = da_prices[18:21] + 120  # Evening price spike

    # aFRR capacity prices (€/MW/h)
    afrr_cap_prices = 25 + 10 * np.abs(np.cos(np.linspace(0, 2 * np.pi, 24))) + np.random.normal(0, 3, 24)

    # Industrial load profile (MW) exhibiting high peaks during midday
    industrial_load = 4.0 + 1.5 * np.sin(np.linspace(0, 2 * np.pi, 24)) + np.random.normal(0, 0.2, 24)
    industrial_load[10:14] = industrial_load[10:14] + 3.0  # Industrial peak hours requiring shaving

    df = pd.DataFrame({
        "Timestamp": hours,
        "DA_Price": da_prices,
        "aFRR_Capacity_Price": afrr_cap_prices,
        "Industrial_Load_MW": industrial_load
    })
    df.set_index("Timestamp", inplace=True)
    return df

market_df = generate_german_market_data()

# ------------------------------------------
# 2. ADVANCED MILP ENGINE WITH PEAK SHAVING PENALTY
# ------------------------------------------
def run_true_co_optimization(df, capacity_mwh=10.0, max_power_mw=5.0, efficiency=0.9, grid_fee_penalty_rate=150.0):
    """
    Solves the MILP co-optimization problem integrating Day-Ahead arbitrage,
    aFRR capacity reservation, and a grid peak-shaving penalty term in the objective function.
    """
    model = plp.LpProblem("German_BESS_True_CoOptimization", plp.LpMaximize)
    time_steps = range(len(df))

    # Decision Variables
    charge = {t: plp.LpVariable(f"charge_{t}", lowBound=0, upBound=max_power_mw) for t in time_steps}
    discharge = {t: plp.LpVariable(f"discharge_{t}", lowBound=0, upBound=max_power_mw) for t in time_steps}
    reserve_afrr = {t: plp.LpVariable(f"reserve_afrr_{t}", lowBound=0, upBound=max_power_mw/2) for t in time_steps}
    soc = {t: plp.LpVariable(f"soc_{t}", lowBound=0, upBound=capacity_mwh) for t in time_steps}

    # Binary variables for charging/discharging mutual exclusivity
    is_charging = {t: plp.LpVariable(f"is_charging_{t}", cat='Binary') for t in time_steps}

    # Auxiliary variable representing the peak grid load to be penalized
    peak_grid_load = plp.LpVariable("peak_grid_load", lowBound=0)

    da_prices = df["DA_Price"].values
    afrr_prices = df["aFRR_Capacity_Price"].values
    ind_load = df["Industrial_Load_MW"].values

    # Objective Function: Maximize Market Revenues MINUS a penalty proportional to the peak grid load
    market_revenue = plp.lpSum([
        (discharge[t] - charge[t]) * da_prices[t] + reserve_afrr[t] * afrr_prices[t]
        for t in time_steps
    ])

    # We maximize total revenue minus the cost/penalty associated with the maximum grid peak
    model += market_revenue - (peak_grid_load * grid_fee_penalty_rate)

    # Constraints
    soc_init = capacity_mwh * 0.5

    for t in time_steps:
        # Net load seen by the grid at step t
        net_grid_load_t = ind_load[t] - discharge[t] + charge[t]

        # Ensure peak_grid_load variable captures the maximum net grid load across all time steps
        model += peak_grid_load >= net_grid_load_t

        # Inverter and reserve power boundaries
        model += charge[t] + reserve_afrr[t] <= max_power_mw
        model += discharge[t] + reserve_afrr[t] <= max_power_mw

        # Mutual exclusivity
        model += charge[t] <= max_power_mw * is_charging[t]
        model += discharge[t] <= max_power_mw * (1 - is_charging[t])

        # SoC dynamics
        if t == 0:
            model += soc[t] == soc_init + (charge[t] * efficiency - discharge[t] / efficiency) * 1.0
        else:
            model += soc[t] == soc[t-1] + (charge[t] * efficiency - discharge[t] / efficiency) * 1.0

    # Final boundary condition for SoC
    model += soc[len(time_steps)-1] >= soc_init

    # Solve via CBC solver
    model.solve(plp.PULP_CBC_CMD(msg=0))

    # Extract results
    results_df = df.copy()
    results_df["Optimized_Charge_MW"] = [plp.value(charge[t]) for t in time_steps]
    results_df["Optimized_Discharge_MW"] = [plp.value(discharge[t]) for t in time_steps]
    results_df["Optimized_Reserve_MW"] = [plp.value(reserve_afrr[t]) for t in time_steps]
    results_df["Optimized_SoC_MWh"] = [plp.value(soc[t]) for t in time_steps]
    results_df["Net_Grid_Load_MW"] = results_df["Industrial_Load_MW"] - results_df["Optimized_Discharge_MW"] + results_df["Optimized_Charge_MW"]

    return results_df

# Run optimized model with peak shaving integration
coopt_results = run_true_co_optimization(market_df)

# Print performance report
original_peak = market_df["Industrial_Load_MW"].max()
optimized_peak = coopt_results["Net_Grid_Load_MW"].max()
print(f"--- TRUE CO-OPTIMIZATION & PEAK SHAVING RESULTS ---")
print(f"Original Peak Load: {original_peak:.2f} MW")
print(f"Optimized Peak Load (Grid-Visible): {optimized_peak:.2f} MW")
print(f"Peak Reduction Achieved: {original_peak - optimized_peak:.2f} MW")

# ------------------------------------------
# 3. VISUALIZATION
# ------------------------------------------
fig = make_subplots(specs=[[{"secondary_y": True}]])
hours = coopt_results.index

fig.add_trace(go.Scatter(x=hours, y=coopt_results['Industrial_Load_MW'], name="Original Load (MW)", line=dict(color='#B6E880', width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=hours, y=coopt_results['Net_Grid_Load_MW'], name="Net Grid Load (MW)", line=dict(color='#EF553B', width=2, dash='dash')), secondary_y=False)
fig.add_trace(go.Bar(x=hours, y=coopt_results['Optimized_Discharge_MW'], name="BESS Discharge (MW)", marker_color='#00CC96'), secondary_y=False)

fig.update_layout(
    title="German BESS Peak Shaving & Grid Fee Avoidance Integration",
    template="plotly_dark",
    hovermode="x unified",
    legend=dict(bgcolor="#581c87", bordercolor="#9333ea", borderwidth=1)
)

fig.show()

--- TRUE CO-OPTIMIZATION & PEAK SHAVING RESULTS ---
Original Peak Load: 7.66 MW
Optimized Peak Load (Grid-Visible): 6.70 MW
Peak Reduction Achieved: 0.96 MW


In [6]:
# ==========================================
# GERMAN BESS CO-OPTIMIZATION & PEAK SHAVING ENGINE (CORE MODULE)
# ==========================================

import io
import os
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pulp as plp


def generate_german_market_data():
  """Generates a 24-hour sample dataset including EPEX Day-Ahead prices,

  German aFRR capacity prices, and industrial load profiles for peak shaving.
  """
  np.random.seed(42)
  hours = pd.date_range(start="2026-06-01", periods=24, freq="h")

  # EPEX Day-Ahead prices (€/MWh) with typical duck-curve / peak behaviors
  da_prices = (
      80
      + 40 * np.sin(np.linspace(0, 2 * np.pi, 24))
      + np.random.normal(0, 15, 24)
  )
  da_prices[18:21] = da_prices[18:21] + 120  # Evening price spike

  # aFRR capacity prices (€/MW/h)
  afrr_cap_prices = (
      25
      + 10 * np.abs(np.cos(np.linspace(0, 2 * np.pi, 24)))
      + np.random.normal(0, 3, 24)
  )

  # Industrial load profile (MW) exhibiting high peaks during midday
  industrial_load = (
      4.0
      + 1.5 * np.sin(np.linspace(0, 2 * np.pi, 24))
      + np.random.normal(0, 0.2, 24)
  )
  industrial_load[10:14] = (
      industrial_load[10:14] + 3.0
  )  # Industrial peak hours requiring shaving

  df = pd.DataFrame({
      "Timestamp": hours,
      "DA_Price": da_prices,
      "aFRR_Capacity_Price": afrr_cap_prices,
      "Industrial_Load_MW": industrial_load,
  })
  df.set_index("Timestamp", inplace=True)
  return df


def run_true_co_optimization(
    df,
    capacity_mwh=10.0,
    max_power_mw=5.0,
    efficiency=0.9,
    grid_fee_penalty_rate=150.0,
):
  """Solves the MILP co-optimization problem integrating Day-Ahead arbitrage,

  aFRR capacity reservation, and a grid peak-shaving penalty term in the
  objective function.
  """
  model = plp.LpProblem("German_BESS_True_CoOptimization", plp.LpMaximize)
  time_steps = range(len(df))

  # Decision Variables
  charge = {
      t: plp.LpVariable(f"charge_{t}", lowBound=0, upBound=max_power_mw)
      for t in time_steps
  }
  discharge = {
      t: plp.LpVariable(f"discharge_{t}", lowBound=0, upBound=max_power_mw)
      for t in time_steps
  }
  reserve_afrr = {
      t: plp.LpVariable(
          f"reserve_afrr_{t}", lowBound=0, upBound=max_power_mw / 2
      )
      for t in time_steps
  }
  soc = {
      t: plp.LpVariable(f"soc_{t}", lowBound=0, upBound=capacity_mwh)
      for t in time_steps
  }

  # Binary variables for charging/discharging mutual exclusivity
  is_charging = {
      t: plp.LpVariable(f"is_charging_{t}", cat="Binary") for t in time_steps
  }

  # Auxiliary variable representing the peak grid load to be penalized
  peak_grid_load = plp.LpVariable("peak_grid_load", lowBound=0)

  da_prices = df["DA_Price"].values
  afrr_prices = df["aFRR_Capacity_Price"].values
  ind_load = df["Industrial_Load_MW"].values

  # Objective Function: Maximize Market Revenues MINUS a penalty proportional to the peak grid load
  market_revenue = plp.lpSum([
      (discharge[t] - charge[t]) * da_prices[t]
      + reserve_afrr[t] * afrr_prices[t]
      for t in time_steps
  ])

  model += market_revenue - (peak_grid_load * grid_fee_penalty_rate)

  # Constraints
  soc_init = capacity_mwh * 0.5

  for t in time_steps:
    net_grid_load_t = ind_load[t] - discharge[t] + charge[t]
    model += peak_grid_load >= net_grid_load_t

    model += charge[t] + reserve_afrr[t] <= max_power_mw
    model += discharge[t] + reserve_afrr[t] <= max_power_mw

    model += charge[t] <= max_power_mw * is_charging[t]
    model += discharge[t] <= max_power_mw * (1 - is_charging[t])

    if t == 0:
      model += (
          soc[t]
          == soc_init
          + (charge[t] * efficiency - discharge[t] / efficiency) * 1.0
      )
    else:
      model += (
          soc[t]
          == soc[t - 1]
          + (charge[t] * efficiency - discharge[t] / efficiency) * 1.0
      )

  model += soc[len(time_steps) - 1] >= soc_init

  # Solve via CBC solver
  model.solve(plp.PULP_CBC_CMD(msg=0))

  # Extract results
  results_df = df.copy()
  results_df["Optimized_Charge_MW"] = [
      plp.value(charge[t]) for t in time_steps
  ]
  results_df["Optimized_Discharge_MW"] = [
      plp.value(discharge[t]) for t in time_steps
  ]
  results_df["Optimized_Reserve_MW"] = [
      plp.value(reserve_afrr[t]) for t in time_steps
  ]
  results_df["Optimized_SoC_MWh"] = [plp.value(soc[t]) for t in time_steps]
  results_df["Net_Grid_Load_MW"] = (
      results_df["Industrial_Load_MW"]
      - results_df["Optimized_Discharge_MW"]
      + results_df["Optimized_Charge_MW"]
  )

  return results_df


def export_results_and_generate_artifacts(results_df):
  """Saves the optimization results to a CSV file, generates static plots,

  and compiles an animated GIF showing the dispatch profile over time.
  """
  # 1. Save to CSV
  csv_path = "german_bess_optimization_results.csv"
  results_df.to_csv(csv_path)
  print(f"Results successfully saved to {csv_path}")

  # 2. Generate Static Graph 1 (Power Profile & Peak Shaving)
  fig1, ax1 = plt.subplots(figsize=(8, 5))
  ax1.plot(
      results_df.index,
      results_df["Industrial_Load_MW"],
      label="Original Load",
      color="#B6E880",
      linewidth=2,
  )
  ax1.plot(
      results_df.index,
      results_df["Net_Grid_Load_MW"],
      label="Net Grid Load",
      color="#EF553B",
      linestyle="--",
      linewidth=2,
  )
  ax1.set_title("BESS Peak Shaving & Grid Load Profile")
  ax1.set_xlabel("Time")
  ax1.set_ylabel("Power (MW)")
  ax1.legend()
  ax1.grid(True, alpha=0.3)
  fig1.savefig("graph1_peak_shaving.png", bbox_inches="tight")
  plt.close(fig1)

  # 3. Generate Static Graph 2 (Market Prices)
  fig2, ax2 = plt.subplots(figsize=(8, 5))
  ax2.plot(
      results_df.index,
      results_df["DA_Price"],
      label="DA Price (€/MWh)",
      color="#FFA15A",
      linewidth=2,
  )
  ax2.set_title("EPEX Day-Ahead Market Prices")
  ax2.set_xlabel("Time")
  ax2.set_ylabel("Price (€/MWh)")
  ax2.legend()
  ax2.grid(True, alpha=0.3)
  fig2.savefig("graph2_market_prices.png", bbox_inches="tight")
  plt.close(fig2)

  # 4. Generate Animated GIF
  frames = []
  for i in range(1, len(results_df) + 1):
    fig_g, ax_g = plt.subplots(figsize=(7, 4))
    ax_g.plot(
        results_df.index[:i],
        results_df["Industrial_Load_MW"][:i],
        label="Original Load",
        color="#B6E880",
    )
    ax_g.plot(
        results_df.index[:i],
        results_df["Net_Grid_Load_MW"][:i],
        label="Net Grid Load",
        color="#EF553B",
        linestyle="--",
    )
    ax_g.set_xlim(results_df.index[0], results_df.index[-1])
    ax_g.set_ylim(0, results_df["Industrial_Load_MW"].max() * 1.2)
    ax_g.set_title(f"BESS Dispatch Animation - Step {i}/24")
    ax_g.set_ylabel("Power (MW)")
    ax_g.grid(True, alpha=0.3)

    frame_buf = io.BytesIO()
    fig_g.savefig(frame_buf, format="png", bbox_inches="tight")
    frame_buf.seek(0)
    frames.append(imageio.imread(frame_buf))
    plt.close(fig_g)

  gif_path = "bess_dispatch_animation.gif"
  imageio.mimsave(gif_path, frames, format="GIF", duration=150)
  print(f"Animated GIF successfully saved to {gif_path}")


# --- EXECUTION PIPELINE ---
if __name__ == "__main__":
  print("Generating German market data...")
  market_data = generate_german_market_data()

  print("Running MILP Co-Optimization & Peak Shaving Engine...")
  optimized_output = run_true_co_optimization(market_data)

  print("Exporting results, charts, and GIF animation...")
  export_results_and_generate_artifacts(optimized_output)

  original_peak = market_data["Industrial_Load_MW"].max()
  optimized_peak = optimized_output["Net_Grid_Load_MW"].max()
  print(
      f"Optimization Complete! Peak Reduced from {original_peak:.2f} MW to"
      f" {optimized_peak:.2f} MW"
  )

Generating German market data...
Running MILP Co-Optimization & Peak Shaving Engine...
Exporting results, charts, and GIF animation...
Results successfully saved to german_bess_optimization_results.csv
Animated GIF successfully saved to bess_dispatch_animation.gif
Optimization Complete! Peak Reduced from 7.66 MW to 6.70 MW
